[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/vision_transformers/blob/main/ddpm/ddpm.ipynb)

In [ ]:
import os
import sys
import subprocess

if "google.colab" in sys.modules and not os.path.exists("unet.py"):
    if not os.path.exists("vision_transformers"):
        subprocess.run(
            ["git", "clone", "https://github.com/a-ramakrishnan/vision_transformers.git"],
            check=True,
        )
    os.chdir("vision_transformers")
    if not os.path.exists("ddpm/unet.py"):
        subprocess.run(["git", "fetch", "origin"], check=True)
        subprocess.run(["git", "checkout", "feat/modular-structure"], check=True)
    os.chdir("ddpm")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [ ]:
import torch
from dataset import get_ddpm_dataloader
from scheduler import LinearNoiseScheduler
from unet import UNet
from train import train_ddpm
from sample import plot_forward_diffusion, sample_ddpm, plot_ddpm_samples

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

BATCH_SIZE = 128
train_loader = get_ddpm_dataloader(batch_size=BATCH_SIZE)

In [ ]:
# =====================================================================
# PART 1: Initialize Scheduler & Visualize Forward Diffusion
# =====================================================================
NUM_TIMESTEPS = 1000
scheduler = LinearNoiseScheduler(
    num_timesteps=NUM_TIMESTEPS,
    beta_start=1e-4,
    beta_end=0.02,
    device=device,
)

plot_forward_diffusion(
    scheduler,
    train_loader,
    timesteps_to_show=[0, 100, 300, 600, 999],
    device=device,
)

In [ ]:
# =====================================================================
# PART 2 & 3: Initialize Time-Conditioned U-Net & Train DDPM
# =====================================================================
NUM_EPOCHS = 20
LR = 1e-4

model = UNet(im_channels=1).to(device)
model = train_ddpm(
    model,
    scheduler,
    train_loader,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    device=device,
    checkpoint_path="ddpm_mnist.pth",
)

In [ ]:
# =====================================================================
# PART 4: Reverse Diffusion Sampling (Generation from Pure Noise)
# =====================================================================
generated_imgs = sample_ddpm(model, scheduler, num_samples=16, device=device)
plot_ddpm_samples(generated_imgs)